<a href="https://colab.research.google.com/github/shahdhesham/Thesis_Set2/blob/main/CodeLLAMA_Set2_OneShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
from huggingface_hub import login
import os

# 1. Read token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Store it into environment variable (optional but helpful)
os.environ["HF_TOKEN"] = hf_token

# 3. Login to HuggingFace Hub
login(token=hf_token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [3]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"GPU device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA NOT available. Using CPU.")

CUDA is available! Using GPU.
GPU device name: NVIDIA A100-SXM4-40GB


In [4]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            83Gi       1.5Gi        77Gi       2.0Mi       4.2Gi        81Gi
Swap:             0B          0B          0B


In [5]:
from google.colab import files
import zipfile
import torch

import os
from transformers import AutoModelForCausalLM, AutoTokenizer

In [6]:
import shutil
import os

# Delete EVERYTHING (folders AND old zips)
!rm -rf input_folder output_folder *.zip

print("✅ All cleaned up!")
print("\n📂 Current directory:")
!ls -la

✅ All cleaned up!

📂 Current directory:
total 16
drwxr-xr-x 1 root root 4096 May 12 13:35 .
drwxr-xr-x 1 root root 4096 May 16 15:38 ..
drwxr-xr-x 4 root root 4096 May 12 13:35 .config
drwxr-xr-x 1 root root 4096 May 12 13:35 sample_data


In [7]:
# 1. Upload ZIP file
print("Upload your ZIP file containing .c files:")
uploaded = files.upload()
zip_name = next(iter(uploaded))

# 2. Extract ZIP
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('input_folder')
print("Files extracted to 'input_folder/'")

Upload your ZIP file containing .c files:


Saving C.zip to C.zip
Files extracted to 'input_folder/'


In [8]:
# 3. Load model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/CodeLlama-7b-Instruct-hf",
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/CodeLlama-7b-Instruct-hf")
tokenizer.pad_token = tokenizer.eos_token  # Add this line
tokenizer.padding_side = "left"  # ✅ Fix right-padding warning



config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

In [9]:
ONE_SHOT_C = """\
#include <stdio.h>
#include <stdlib.h>
#define MAX 100
struct Node {
    int data ;
    struct Node * left , * right ;
} ;
struct Node * newNode ( int data ) {
    struct Node * node = ( struct Node * ) malloc ( sizeof ( struct Node ) ) ;
    node -> data = data ;
    node -> left = node -> right = NULL ;
    return node ;
}
int findMax ( int arr [ ] , int n ) {
    int max = arr [ 0 ] ;
    for ( int i = 1 ; i < n ; i ++ ) {
        if ( arr [ i ] > max )
            max = arr [ i ] ;
    }
    return max ;
}
int main ( ) {
    int arr [ ] = { 3 , 1 , 4 , 1 , 5 , 9 , 2 , 6 } ;
    int n = sizeof ( arr ) / sizeof ( arr [ 0 ] ) ;
    int result = findMax ( arr , n ) ;
    printf ( "Maximum value is %d\n" , result ) ;
    struct Node * root = newNode ( 1 ) ;
    root -> left = newNode ( 2 ) ;
    root -> right = newNode ( 3 ) ;
    getchar ( ) ;
    return 0 ;
}
"""
ONE_SHOT_CPP = """\
#include <iostream>
using namespace std ;
#define MAX 100
struct Node {
    int data ;
    Node * left , * right ;
} ;
Node * newNode ( int data ) {
    Node * node = new Node ( ) ;
    node -> data = data ;
    node -> left = node -> right = nullptr ;
    return node ;
}
int findMax ( int arr [ ] , int n ) {
    int max = arr [ 0 ] ;
    for ( int i = 1 ; i < n ; i ++ ) {
        if ( arr [ i ] > max )
            max = arr [ i ] ;
    }
    return max ;
}
int main ( ) {
    int arr [ ] = { 3 , 1 , 4 , 1 , 5 , 9 , 2 , 6 } ;
    int n = sizeof ( arr ) / sizeof ( arr [ 0 ] ) ;
    int result = findMax ( arr , n ) ;
    cout << "Maximum value is " << result << "\n" ;
    Node * root = newNode ( 1 ) ;
    root -> left = newNode ( 2 ) ;
    root -> right = newNode ( 3 ) ;
    return 0 ;
}
"""

def translate_batch(c_code_list):
    all_messages = []
    for c_code in c_code_list:
        system_prompt = """You are an expert code translator. Your ONLY task is to convert C code to C++ code.
Rules you MUST follow:
1. Output ONLY executable C++ code
2. Never include markdown or explanations
3. Preserve all functionality exactly
4. Use standard C++ libraries
5. Match the original code's input/output behavior."""

        one_shot_user_prompt = f"""Translate this C code to C++ code:

C Code:
{ONE_SHOT_C}

C++ Code:"""

        user_prompt = f"""Translate this C code to C++ code:

C Code:
{c_code}

C++ Code:"""

        messages = [
            {"role": "system",    "content": system_prompt},
            # ── ONE-SHOT EXAMPLE ──────────────────────────
            {"role": "user",      "content": one_shot_user_prompt},
            {"role": "assistant", "content": ONE_SHOT_CPP},
            # ── ACTUAL INPUT ──────────────────────────────
            {"role": "user",      "content": user_prompt}
        ]
        all_messages.append(messages)

    # Get TEXT strings
    input_texts = []
    for messages in all_messages:
        text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False
        )
        input_texts.append(text)

    # Tokenize with padding
    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False
    ).to(model.device)

    # ── CodeLlama terminator — simpler than Llama ──
    terminators = tokenizer.eos_token_id

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        eos_token_id=terminators,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=False,
    )

    # Extract generated tokens
    results = []
    for i, output in enumerate(outputs):
        input_len = inputs['attention_mask'][i].sum().item()
        response = output[input_len:]
        cpp_code = tokenizer.decode(response, skip_special_tokens=True)
        results.append(cpp_code.strip())

    return results

In [10]:
#batching
batch_size = 4
batch_files = []
batch_codes = []
batch_paths = []

for root, _, files in os.walk('input_folder'):
    for file in files:
        if file.endswith('.c'):
            in_path = os.path.join(root, file)
            out_path = in_path.replace('input_folder', 'output_folder').replace('.c', '.cpp')
            os.makedirs(os.path.dirname(out_path), exist_ok=True)

            with open(in_path, 'r') as f:
                code = f.read()

            batch_files.append(file)
            batch_codes.append(code)
            batch_paths.append((in_path, out_path))

            # Once batch is full, translate all at once
            if len(batch_codes) == batch_size:
                translations = translate_batch(batch_codes)
                for (in_p, out_p), translation in zip(batch_paths, translations):
                    with open(out_p, 'w') as f_out:
                        f_out.write(translation)
                    print(f"Translated: {in_p} → {out_p}")

                import gc
                gc.collect()
                torch.cuda.empty_cache()
                # Clear batch lists
                batch_files = []
                batch_codes = []
                batch_paths = []




# Translate any remaining files smaller than batch size
if batch_codes:
    translations = translate_batch(batch_codes)
    for (in_p, out_p), translation in zip(batch_paths, translations):
        with open(out_p, 'w') as f_out:
            f_out.write(translation)
        print(f"Translated: {in_p} → {out_p}")

Translated: input_folder/C/2048.c → output_folder/C/2048.cpp
Translated: input_folder/C/13545.c → output_folder/C/13545.cpp
Translated: input_folder/C/723.c → output_folder/C/723.cpp
Translated: input_folder/C/7320.c → output_folder/C/7320.cpp
Translated: input_folder/C/13653.c → output_folder/C/13653.cpp
Translated: input_folder/C/8947.c → output_folder/C/8947.cpp
Translated: input_folder/C/12814.c → output_folder/C/12814.cpp
Translated: input_folder/C/2074.c → output_folder/C/2074.cpp
Translated: input_folder/C/13513.c → output_folder/C/13513.cpp
Translated: input_folder/C/9367.c → output_folder/C/9367.cpp
Translated: input_folder/C/139.c → output_folder/C/139.cpp
Translated: input_folder/C/638.c → output_folder/C/638.cpp
Translated: input_folder/C/2415.c → output_folder/C/2415.cpp
Translated: input_folder/C/10289.c → output_folder/C/10289.cpp
Translated: input_folder/C/1610.c → output_folder/C/1610.cpp
Translated: input_folder/C/1701.c → output_folder/C/1701.cpp
Translated: input_fo

In [11]:
from google.colab import files as colab_files  # CHANGED: Added alias

In [12]:
# 6. Compress and download
print("\nCreating output ZIP...")
!zip -r CodeLLama_Set2_OneShot_output.zip output_folder
colab_files.download('CodeLLama_Set2_OneShot_output.zip')  # CHANGED: Uses alias
print("Done! Download should start automatically.")


Creating output ZIP...
  adding: output_folder/ (stored 0%)
  adding: output_folder/C/ (stored 0%)
  adding: output_folder/C/13414.cpp (deflated 64%)
  adding: output_folder/C/1710.cpp (deflated 56%)
  adding: output_folder/C/13537.cpp (deflated 58%)
  adding: output_folder/C/264.cpp (deflated 73%)
  adding: output_folder/C/2142.cpp (deflated 68%)
  adding: output_folder/C/9094.cpp (deflated 66%)
  adding: output_folder/C/1973.cpp (deflated 65%)
  adding: output_folder/C/8760.cpp (deflated 60%)
  adding: output_folder/C/1623.cpp (deflated 61%)
  adding: output_folder/C/1687.cpp (deflated 57%)
  adding: output_folder/C/10557.cpp (deflated 52%)
  adding: output_folder/C/2538.cpp (deflated 61%)
  adding: output_folder/C/1705.cpp (deflated 55%)
  adding: output_folder/C/9703.cpp (deflated 60%)
  adding: output_folder/C/2.cpp (deflated 58%)
  adding: output_folder/C/2001.cpp (deflated 60%)
  adding: output_folder/C/9497.cpp (deflated 64%)
  adding: output_folder/C/7321.cpp (deflated 58%)
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Download should start automatically.


In [13]:
print(model.generation_config)


GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 2
}

